In [1]:
#MERGE, DEDUPLICATE, CREATE TABLE, CREATE INDAGATE POINT.
#if you run the first cell you need to adjust the paths.


"""
Merge multiple output sets from the LPG Africa search script.
Reads CSV, JSON, and GPKG files from different runs,
deduplicates by place_id, and writes combined outputs:
  - all_africa_combined.csv
  - all_africa_combined.json
  - all_africa_combined_filling_3857.gpkg
  - all_africa_combined_reseller_3857.gpkg
plus a new summary_combined.csv

"""

import os
import csv
import json
import glob
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from collections import Counter

# ================== CONFIGURATION ==================
OUTPUT_ROOT = "location_total"          # folder where combined files will be saved
# List of directories containing the raw output files.
# The script will process all files matching the patterns inside these directories.
INPUT_DIRS = [
    "location_total/run_parziali",
]

# ===================================================

def find_unique_files(dirs, pattern):
    """Return a sorted list of unique absolute file paths matching pattern in dirs."""
    files = set()
    for d in dirs:
        full_pattern = os.path.join(d, pattern)
        for fpath in glob.glob(full_pattern):
            files.add(os.path.abspath(fpath))
    return sorted(files)

def load_place_ids(dirs, pattern):
    """Load place_ids from a specific CSV file in one of the given directories."""
    ids = set()
    for d in dirs:
        filepath = os.path.join(d, pattern)
        if os.path.exists(filepath):
            print(f"Reading place_ids from: {filepath}")
            with open(filepath, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    ids.add(row['place_id'])
            return ids
    print(f"File {pattern} not found in any directory")
    return ids

def load_csv_files(files):
    """Load records from list of CSV files."""
    records = []
    for fpath in files:
        print(f"Reading CSV: {fpath}")
        with open(fpath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                try:
                    row['lat'] = float(row['lat'])
                    row['lng'] = float(row['lng'])
                    row['user_ratings_total'] = int(row['user_ratings_total'])
                    row['rating'] = float(row['rating']) if row['rating'] else None
                except:
                    pass
                records.append(row)
    return records

def load_json_files(files):
    """Load records from list of JSON files."""
    records = []
    for fpath in files:
        print(f"Reading JSON: {fpath}")
        with open(fpath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            records.extend(data)
    return records

def load_gpkg_files(files):
    """Load records from GPKG files, handling filling/reseller layers based on filename."""
    records = []
    for fpath in files:
        print(f"Reading GPKG: {fpath}")
        try:
            basename = os.path.basename(fpath)
            # Determine layer to read from filename
            if '_filling_' in basename:
                layer = 'filling'
            elif '_reseller_' in basename:
                layer = 'reseller'
            else:
                # Fallback: try to read both layers (if any)
                for lyr in ['filling', 'reseller']:
                    try:
                        gdf = gpd.read_file(fpath, layer=lyr)
                        if not gdf.empty:
                            gdf = gdf.to_crs("EPSG:4326")
                            gdf['lat'] = gdf.geometry.y
                            gdf['lng'] = gdf.geometry.x
                            recs = gdf.drop(columns='geometry').to_dict('records')
                            for r in recs:
                                r['category'] = lyr
                                r['source'] = r.get('source', 'osm')
                                r['language'] = r.get('language', '')
                                r['keyword'] = r.get('keyword', 'osm')
                            records.extend(recs)
                    except:
                        pass
                continue

            # Read the specific layer
            gdf = gpd.read_file(fpath, layer=layer)
            if gdf.empty:
                continue
            gdf = gdf.to_crs("EPSG:4326")
            gdf['lat'] = gdf.geometry.y
            gdf['lng'] = gdf.geometry.x
            recs = gdf.drop(columns='geometry').to_dict('records')
            for r in recs:
                r['category'] = layer
                r['source'] = r.get('source', 'osm')
                r['language'] = r.get('language', '')
                r['keyword'] = r.get('keyword', 'osm')
            records.extend(recs)

        except Exception as e:
            print(f"Warning: could not read {fpath} - {e}")

    return records

def merge_and_deduplicate(records):
    """Merge records, keep filling over reseller if same place_id."""
    merged = {}
    for rec in records:
        pid = rec['place_id']
        if pid not in merged:
            merged[pid] = rec
        elif rec.get('category') == 'filling' and merged[pid].get('category') != 'filling':
            merged[pid] = rec
    return list(merged.values())

def save_combined_files(results, output_root):
    base = os.path.join(output_root, "all_africa_combined")
    fieldnames = ['place_id','name','address','lat','lng','types','keyword',
                  'search_lat','search_lon','rating','user_ratings_total',
                  'source','language','category']

    # CSV
    with open(f"{base}.csv", 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    print(f"CSV saved: {base}.csv")

    # JSON
    with open(f"{base}.json", 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"JSON saved: {base}.json")

    # GeoPackage
    keep_cols = ['place_id','name','address','types','keyword','search_lat','search_lon',
                 'rating','user_ratings_total','source','language','category']
    filling = [r for r in results if r['category'] == 'filling']
    reseller = [r for r in results if r['category'] == 'reseller']
    if filling:
        df_f = pd.DataFrame(filling)
        geom_f = [Point(lon, lat) for lon, lat in zip(df_f['lng'], df_f['lat'])]
        gdf_f = gpd.GeoDataFrame(df_f[[c for c in keep_cols if c in df_f.columns]],
                                 geometry=geom_f, crs="EPSG:4326")
        gdf_f['lat'] = df_f['lat']; gdf_f['lon'] = df_f['lng']
        gdf_f.to_crs("EPSG:3857").to_file(f"{base}_filling_3857.gpkg", driver='GPKG', layer='filling')
        print(f"GPKG filling saved: {base}_filling_3857.gpkg")
    if reseller:
        df_r = pd.DataFrame(reseller)
        geom_r = [Point(lon, lat) for lon, lat in zip(df_r['lng'], df_r['lat'])]
        gdf_r = gpd.GeoDataFrame(df_r[[c for c in keep_cols if c in df_r.columns]],
                                 geometry=geom_r, crs="EPSG:4326")
        gdf_r['lat'] = df_r['lat']; gdf_r['lon'] = df_r['lng']
        gdf_r.to_crs("EPSG:3857").to_file(f"{base}_reseller_3857.gpkg", driver='GPKG', layer='reseller')
        print(f"GPKG reseller saved: {base}_reseller_3857.gpkg")

def generate_summary(results, output_root):
    total_osm = sum(1 for r in results if r['source'] == 'osm')
    total_google = sum(1 for r in results if r['source'] == 'google')
    total_filling = sum(1 for r in results if r['category'] == 'filling')
    total_reseller = sum(1 for r in results if r['category'] == 'reseller')
    google_recs = [r for r in results if r['source'] == 'google']
    lang_counter = Counter(r['language'] for r in google_recs) if google_recs else Counter()
    lang_str = ", ".join(f"{l}:{cnt}" for l, cnt in lang_counter.most_common()) if lang_counter else ""
    summary_row = {
        'Region': 'Sub-Saharan Africa',
        'Total_Points': len(results),
        'Total_OSM': total_osm,
        'Total_Google': total_google,
        'Total_Filling': total_filling,
        'Total_Reseller': total_reseller,
        'Google_Languages': lang_str
    }
    summary_path = os.path.join(output_root, "summary_combined.csv")
    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        fieldnames = ['Region','Total_Points','Total_OSM','Total_Google',
                      'Total_Filling','Total_Reseller','Google_Languages']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow(summary_row)
    print(f"Summary saved: {summary_path}")

def main():
    # Find unique files
    csv_files = find_unique_files(INPUT_DIRS, "all_africa_combined_*.csv")
    json_files = find_unique_files(INPUT_DIRS, "all_africa_combined_*.json")
    gpkg_files = find_unique_files(INPUT_DIRS, "all_africa_combined_*_3857.gpkg")

    # Load records
    csv_recs = load_csv_files(csv_files)
    json_recs = load_json_files(json_files)
    gpkg_recs = load_gpkg_files(gpkg_files)

    all_recs = csv_recs + json_recs + gpkg_recs
    print(f"\nTotal raw records: {len(all_recs)}")
    merged = merge_and_deduplicate(all_recs)
    print(f"After dedup by place_id: {len(merged)}")

    save_combined_files(merged, OUTPUT_ROOT)
    generate_summary(merged, OUTPUT_ROOT)
    print("Merge completed.")

# ================== OPTIONAL: COMPARE TWO RUNS ==================
if __name__ == "__main__":
    # Optional: compare two specific runs
    # Uncomment to test:
    # run1_ids = load_place_ids(INPUT_DIRS, "all_africa_combined_20260503_100016.csv")
    # run2_ids = load_place_ids(INPUT_DIRS, "all_africa_combined_20260504_114901.csv")
    # common = run1_ids & run2_ids
    # only_run1 = run1_ids - run2_ids
    # only_run2 = run2_ids - run1_ids
    # print(f"Run 1 unique: {len(run1_ids)}")
    # print(f"Run 2 unique: {len(run2_ids)}")
    # print(f"Common: {len(common)}")
    # print(f"Only in run1: {len(only_run1)}")
    # print(f"Only in run2: {len(only_run2)}")
    # print(f"Theoretical union: {len(run1_ids | run2_ids)}")

    # Run main merge
    main()


Reading CSV: c:\Users\matti\OneDrive - Politecnico di Milano\Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri\location_total\run_parziali\all_africa_combined_20260503_100016.csv
Reading CSV: c:\Users\matti\OneDrive - Politecnico di Milano\Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri\location_total\run_parziali\all_africa_combined_20260504_114901.csv
Reading JSON: c:\Users\matti\OneDrive - Politecnico di Milano\Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri\location_total\run_parziali\all_africa_combined_20260503_100016.json
Reading JSON: c:\Users\matti\OneDrive - Politecnico di Milano\Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri\location_total\run_parziali\all_africa_combined_20260504_114901.json
Reading GPKG: c:\Users\matti\OneDrive - Politecnico di Milano\Thesis_onstoveM&F\OnStoveThesis\thesis\script_nostri\location_total\run_parziali\all_africa_combined_20260503_100016_filling_3857.gpkg
Reading GPKG: c:\Users\matti\OneDrive - Politecnico di Milano\Thesis_on

In [ ]:
#!/usr/bin/env python3
"""
Script to produce country summary table and a coverage raster for LPG points
in sub‑Saharan Africa.
Inputs:
  - all_africa_combined.csv (combined points)
  - world_bound/ne_110m_admin_0_countries.shp (country boundaries)
  - run_parziali/ (folder with processed_points.txt files)
Outputs:
  - location_total/summary_countries.csv
  - location_total/coverage_mask.tif
Prints ranking by total points per country.
"""

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import rasterio
from rasterio.transform import from_bounds
from collections import defaultdict
import glob

# ================== CONFIGURATION ==================
BASE_DIR = "location_total"
CSV_INPUT = os.path.join(BASE_DIR, "all_africa_combined.csv")
SHP_PATH = os.path.join(BASE_DIR, "world_bound", "ne_110m_admin_0_countries.shp")
OUTPUT_CSV = os.path.join(BASE_DIR, "summary_countries.csv")
OUTPUT_TIF = os.path.join(BASE_DIR, "coverage_mask.tif")
PROCESSED_DIR = os.path.join(BASE_DIR, "run_parziali")


# Parameters from the original search script (must match)
STEP = 0.65
RADIUS = 50000

# SSA_COUNTRIES dictionary keys (lowercase) – same as in the search script
SSA_COUNTRIES = {
    'angola', 'benin', 'botswana', 'burkina faso', 'burundi', 'cameroon',
    'central african republic', 'chad', 'republic of the congo',
    'democratic republic of the congo', 'ivory coast', 'djibouti',
    'equatorial guinea', 'eritrea', 'eswatini', 'ethiopia', 'gabon',
    'gambia', 'ghana', 'guinea', 'guinea-bissau', 'kenya', 'lesotho',
    'liberia', 'madagascar', 'malawi', 'mali', 'mauritania', 'mozambique',
    'namibia', 'niger', 'nigeria', 'rwanda', 'senegal', 'sierra leone',
    'somalia', 'south africa', 'south sudan', 'sudan',
    'united republic of tanzania', 'togo', 'uganda', 'zambia', 'zimbabwe'
}

# ================== 1. COUNTRY SUMMARY (grid‑based assignment) ==================
def rebuild_search_grid():
    """
    Recreate the exact same grid that the search script used.
    Returns a dict (lat_rounded, lon_rounded) -> set of country names.
    """
    import os, math
    from shapely.geometry import Point

    # ---- functions identical to the original all_africa script ----
    def load_country_geometries():
        world = gpd.read_file(SHP_PATH)
        # Identify the name column
        for col in ['ADMIN', 'NAME', 'SOVEREIGNT']:
            if col in world.columns:
                name_col = col
                break
        else:
            raise KeyError("No country name column found.")
        world['name_lower'] = world[name_col].str.lower()
        ssa_gdf = world[world['name_lower'].isin(SSA_COUNTRIES)].copy()
        if ssa_gdf.empty:
            raise ValueError("No SSA countries matched in shapefile.")
        return {row['name_lower']: row.geometry for _, row in ssa_gdf.iterrows()}, world.crs

    def buffer_geometry(geom, buffer_deg):
        return geom.buffer(buffer_deg)

    def generate_grid(geom, step):
        minx, miny, maxx, maxy = geom.bounds
        points = []
        lat = miny
        while lat <= maxy:
            lon = minx
            while lon <= maxx:
                pt = Point(lon, lat)
                if geom.contains(pt):
                    points.append((round(lat, 6), round(lon, 6)))
                lon += step
            lat += step
        return points

    country_geoms, _ = load_country_geometries()
    buffer_deg = 50.0 / 111.0   # ~0.45045°, same as original
    all_raw = []
    for country, geom in country_geoms.items():
        buffered = buffer_geometry(geom, buffer_deg)
        pts = generate_grid(buffered, STEP)
        for lat, lon in pts:
            all_raw.append((lat, lon, country))

    # deduplicate as in original: same rounding to 4 decimals, merge countries
    from collections import defaultdict
    merged = defaultdict(set)
    for lat, lon, cntry in all_raw:
        key = (round(lat, 4), round(lon, 4))
        merged[key].add(cntry)
    return dict(merged)

def generate_country_summary():
    print("Assigning points using search grid...")
    grid = rebuild_search_grid()       # (lat,lon) -> set of country names
    print(f"Search grid rebuilt: {len(grid)} unique search points.")

    # Load combined results
    df = pd.read_csv(CSV_INPUT)
    # Keep only needed columns
    cols = ['place_id', 'lat', 'lng', 'source', 'category', 'search_lat', 'search_lon']
    df = df[cols].copy()

    # Map each result to a list of countries based on its search point
    def get_countries(row):
        key = (round(row['search_lat'], 4), round(row['search_lon'], 4))
        return grid.get(key, {'unknown'})

    df['countries'] = df.apply(get_countries, axis=1)

    # Expand rows (one row per assigned country)
    rows = []
    for _, rec in df.iterrows():
        for c in rec['countries']:
            rows.append({'place_id': rec['place_id'],
                         'source': rec['source'],
                         'category': rec['category'],
                         'country': c})
    expanded = pd.DataFrame(rows)

    print("Generating summary...")
    stats = expanded.groupby('country').agg(
        Total_Points=('place_id', 'count'),
        OSM=('source', lambda x: (x == 'osm').sum()),
        Google=('source', lambda x: (x == 'google').sum()),
        Filling=('category', lambda x: (x == 'filling').sum()),
        Reseller=('category', lambda x: (x == 'reseller').sum())
    ).reset_index()

    stats['%_OSM'] = (stats['OSM'] / stats['Total_Points'] * 100).round(2)
    stats['%_Google'] = (stats['Google'] / stats['Total_Points'] * 100).round(2)

    stats_sorted = stats.sort_values('country')
    stats_sorted.to_csv(OUTPUT_CSV, index=False)
    print(f"Country summary saved to {OUTPUT_CSV}")

    # Print ranking
    ranking = stats_sorted.sort_values('Total_Points', ascending=False)
    print("\nCountry ranking (total points):")
    for _, row in ranking.iterrows():
        print(f"{row['country']}: {row['Total_Points']}")

    # No more unmatched points – all are accounted for through the grid.
    total_records = len(df) * df['countries'].apply(len).sum()
    print(f"\nTotal point‑country assignments: {total_records}")


# ================== 2. RASTER COVERAGE (true 50 km circles) ==================
def generate_coverage_raster():
    print("\nCollecting search points from run_parziali/...")
    pattern = os.path.join(PROCESSED_DIR, "**", "processed_points.txt")
    files = glob.glob(pattern, recursive=True)

    if not files:
        print("No processed_points.txt files found. Coverage raster will be empty.")
        return

    points_set = set()
    for fpath in files:
        with open(fpath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    lat_str, lon_str = line.split(',')
                    lat = float(lat_str)
                    lon = float(lon_str)
                    key = (round(lat, 4), round(lon, 4))
                    points_set.add(key)
                except ValueError:
                    continue
    print(f"Found {len(points_set)} unique search points.")

    # Get Africa extent (same as before)
    world = gpd.read_file(SHP_PATH)
    africa = world[world['CONTINENT'] == 'Africa']
    if africa.empty:
        raise ValueError("No African countries found in shapefile (field 'CONTINENT' missing?).")
    minx, miny, maxx, maxy = africa.total_bounds

    pixel_size = 0.01   # ~1 km at the equator
    width = max(1, int(np.ceil((maxx - minx) / pixel_size)))
    height = max(1, int(np.ceil((maxy - miny) / pixel_size)))
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    # Convert search points to arrays
    search_lats = np.array([lat for lat, lon in points_set])
    search_lons = np.array([lon for lat, lon in points_set])

    # Create pixel centres
    rows, cols = np.indices((height, width))
    pixel_lons = minx + (cols + 0.5) * pixel_size
    pixel_lats = maxy - (rows + 0.5) * pixel_size

    # Use a fast distance approach with cKDTree if available,
    # otherwise fall back to slow but correct circle drawing.
    try:
        from scipy.spatial import cKDTree
        # Convert everything to kilometres (plate carrée approximation)
        R = 6371.0
        avg_lat = np.radians((miny + maxy) / 2)
        search_x = np.deg2rad(search_lons) * R * np.cos(avg_lat)
        search_y = np.deg2rad(search_lats) * R
        pixel_x = np.deg2rad(pixel_lons) * R * np.cos(avg_lat)
        pixel_y = np.deg2rad(pixel_lats) * R

        tree = cKDTree(np.column_stack((search_x, search_y)))
        dist, _ = tree.query(np.column_stack((pixel_x.ravel(), pixel_y.ravel())), k=1)
        dist = dist.reshape(height, width)
        raster = (dist <= RADIUS/1000.0).astype(np.uint8)   # RADIUS is in metres
        print("Coverage computed with cKDTree (fast).")
    except ImportError:
        print("SciPy not available, using slower circle method...")
        radius_deg = RADIUS / (111_000 * np.cos(np.radians(avg_lat)))   # deg per metre
        raster = np.zeros((height, width), dtype=np.uint8)
        for lat, lon in points_set:
            lat_span = radius_deg
            lon_span = radius_deg
            r1 = max(0, int((maxy - lat - lat_span) / pixel_size))
            r2 = min(height, int((maxy - lat + lat_span) / pixel_size) + 1)
            c1 = max(0, int((lon - lon_span - minx) / pixel_size))
            c2 = min(width, int((lon + lon_span - minx) / pixel_size) + 1)
            for r in range(r1, r2):
                for c in range(c1, c2):
                    pix_lat = maxy - (r + 0.5) * pixel_size
                    pix_lon = minx + (c + 0.5) * pixel_size
                    # Approximate distance in degrees (spherical earth)
                    dlat = abs(pix_lat - lat)
                    dlon = abs(pix_lon - lon) * np.cos(np.radians((pix_lat + lat)/2))
                    if np.hypot(dlat, dlon) <= radius_deg:
                        raster[r, c] = 1

    print(f"Creating coverage raster ({width}x{height})...")
    with rasterio.open(
        OUTPUT_TIF, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype='uint8',
        crs='EPSG:4326',
        transform=transform,
        compress='lzw',
        nodata=0
    ) as dst:
        dst.write(raster, 1)
    print(f"Coverage raster saved to {OUTPUT_TIF}")


# ================== MAIN ==================
if __name__ == "__main__":
    if not os.path.exists(CSV_INPUT):
        print(f"Error: input file {CSV_INPUT} not found.")
        exit(1)
    generate_country_summary()
    generate_coverage_raster()

#TODO ADJUST DECOUPLING ONSHORE E OFF SHORE, REAL COUNTRY OWNERSHIP. INCLUDE 0 POINTS COUNTRY.

Assigning points using search grid...
Search grid rebuilt: 6164 unique search points.
Generating summary...
Country summary saved to location_total\summary_countries.csv

Country ranking (total points):
south africa: 407
united republic of tanzania: 150
ghana: 115
cameroon: 113
ivory coast: 103
angola: 40
benin: 40
nigeria: 36
kenya: 32
uganda: 21
burkina faso: 18
zambia: 18
niger: 15
mozambique: 12
sierra leone: 7
gabon: 7
ethiopia: 7
eswatini: 7
sudan: 6
democratic republic of the congo: 6
senegal: 5
namibia: 5
lesotho: 5
malawi: 5
botswana: 4
zimbabwe: 3
guinea: 2
madagascar: 2
liberia: 1
gambia: 1
republic of the congo: 1
mali: 1
mauritania: 1
south sudan: 1

Total point‑country assignments: 1432809

Found 6164 unique search points.
Coverage computed with cKDTree (fast).
Creating coverage raster (6876x7217)...
Coverage raster saved to location_total\coverage_mask.tif
